# Using PySpark with TimeCopilot

When using PySpark Java needs to be installed with a `JAVA_HOME` environment variable set. The environment variable may have been set when installing Java.

The required Java version may vary depending on the PySpark version in use.
For this example PySpark version `4.0.2` was used with OpenJDK version `21.0.10` used as the Java version.

In [1]:
import nest_asyncio

nest_asyncio.apply()

from timecopilot import TimeCopilotForecaster

from pyspark.sql import SparkSession
import pandas as pd

from timecopilot.models import SeasonalNaive
from timecopilot.models.foundation.chronos import Chronos

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


checking the `JAVA_HOME` environment variable.

the path below for a version of openjdk installed via homebrew on a mac

In [2]:
import os
print(os.environ['JAVA_HOME'])

/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home


## Start the Spark session

In [3]:

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 15:41:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Create the dataframe

In [4]:
df = pd.read_csv("https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv", parse_dates=["ds"])
s_df = spark.createDataFrame(df)
print(s_df)
s_df.show(n=5)

DataFrame[unique_id: string, ds: timestamp, y: bigint]
+-----------+-------------------+-----+
|  unique_id|                 ds|    y|
+-----------+-------------------+-----+
|Oktoberfest|2020-01-31 00:00:00|25376|
|Oktoberfest|2020-02-29 00:00:00|28470|
|Oktoberfest|2020-03-31 00:00:00|23816|
|Oktoberfest|2020-04-30 00:00:00|46186|
|Oktoberfest|2020-05-31 00:00:00|31213|
+-----------+-------------------+-----+
only showing top 5 rows


## Create a TimeCopilotForecaster

In [5]:
tcf = TimeCopilotForecaster(
    models=[
        SeasonalNaive(),
        Chronos(repo_id="autogluon/chronos-2-small")
    ]
)

## Create a forecast

In [7]:
result = tcf.forecast(
    df=s_df,
    h=12
)

2026-03-23 15:42:30,213	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-23 15:42:30,268	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## View forecast results

In [9]:
print(result)
result.show(n=72)

DataFrame[unique_id: string, ds: timestamp, SeasonalNaive: double, Chronos: double]


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
  0%|          | 0/1 [00:00<?, ?it/s]

+------------+-------------------+-------------+-----------------+
|   unique_id|                 ds|SeasonalNaive|          Chronos|
+------------+-------------------+-------------+-----------------+
|Black Friday|2025-09-30 00:00:00|       2607.0| 1740.37158203125|
|Black Friday|2025-10-31 00:00:00|       2470.0| 2226.00732421875|
|Black Friday|2025-11-30 00:00:00|      11058.0|   12542.42578125|
|Black Friday|2025-12-31 00:00:00|       3548.0|  3203.5224609375|
|Black Friday|2026-01-31 00:00:00|       1724.0| 1596.01318359375|
|Black Friday|2026-02-28 00:00:00|       1730.0|1594.218017578125|
|Black Friday|2026-03-31 00:00:00|       1874.0|1539.522705078125|
|Black Friday|2026-04-30 00:00:00|       2311.0|1530.237548828125|
|Black Friday|2026-05-31 00:00:00|       1332.0|  1492.1279296875|
|Black Friday|2026-06-30 00:00:00|       1215.0| 1452.61865234375|
|Black Friday|2026-07-31 00:00:00|       1108.0| 1441.93115234375|
|Black Friday|2026-08-31 00:00:00|       1690.0| 1470.73583984

100%|██████████| 1/1 [00:00<00:00,  9.61it/s]


/Users/shane/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
